# Knowledge-editing AMD-Llama-135M with MARV (Colab T4 or CPU)

`amd/AMD-Llama-135m` is a Llama-2-architecture model (12 layers, hidden 768,
intermediate 2048, Llama-2 tokenizer) trained on ~670B tokens of SlimPajama +
Project Gutenberg. It loads as `LlamaForCausalLM`, so MARV's `LlamaStyleFFN`
adapter handles it with no special-casing.

**Why this model for an editing demo:** it is tiny. Extraction is instant,
`build_down_meta` over its 32k vocab is instant on CPU, and a **300-probe
battery runs in seconds** -- which is the whole point of the exercise below:
you cannot estimate collateral damage from 8 control probes, and here you can
afford 120+.

**Honest caveats.** It is a *base* model (no chat template -- every prompt here
is plain completion) with weak factual recall (MMLU at chance). Pick a
high-frequency fact like `France -> Paris`; its constellation will be smaller
and noisier than on a 1B model. The method is what transfers, not the exact
numbers.

Also runs the base-vs-code weight diff: `amd/AMD-Llama-135m` vs
`amd/AMD-Llama-135m-code` (StarCoder-Python finetune, same 12 layers).

Runtime: T4 GPU is fine; CPU works too (this model is small enough).

In [ ]:
!pip install -q 'transformers>=4.40' accelerate safetensors matplotlib
!git clone -q https://github.com/thebnbrkr/marv.git /content/marv
%cd /content/marv
!pip install -q -e .

## Load + extract

In [ ]:
import math, torch, numpy as np, gc, marv
from transformers import LlamaForCausalLM, AutoTokenizer
device = 'cuda' if torch.cuda.is_available() else 'cpu'

BASE = 'amd/AMD-Llama-135m'
CODE = 'amd/AMD-Llama-135m-code'

tok = AutoTokenizer.from_pretrained(BASE)
model = LlamaForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16 if device=='cuda' else torch.float32).to(device).eval()
print(model.config.num_hidden_layers, 'layers  hidden', model.config.hidden_size,
      ' intermediate', model.config.intermediate_size, ' vocab', model.config.vocab_size)

vindex = marv.extract(model, model_name=BASE)
marv.build_down_meta(vindex, device=device)      # 32k vocab -> instant
vindex.save('/content/amd-135m-base.vindex.npz')
print('bands:', vindex.layer_bands)

## Browse + locate the constellation

`describe_entity` is the bare-embedding view; `constellation(..., model=)` queries
the real hidden state (differenced against a baseline), which is sharper.

In [ ]:
for entity in ['France', 'Paris', 'Germany']:
    print(f'=== {entity} ===')
    for r in marv.describe_entity(vindex, tok, entity, k_features=3):
        print('  ', r)

pool = marv.constellation(vindex, tok, 'France', model=model,
                          prompt='The capital of France is',
                          baseline_prompt='The capital of',
                          per_layer=8, device=device)
print('\ncontextual constellation (top 12):')
for r in pool[:12]:
    print(f'  L{r.layer:>2} f{r.feature:<5} sim={r.sim:.2f}  -> {r.tokens[:3]}')

## Why 8 controls is not enough

Collateral rate is a **proportion**; its standard error is `sqrt(p(1-p)/n)`.
At n=8 and a true 10% rate the error bar (+/- 0.11) is bigger than the number
you're trying to measure. Below: the *same edit*, scored against 8 controls and
against `marv.broad_controls()` (120 probes across 6 sub-domains).

`marv.capital_edit_battery` builds the wide one: 4 target rephrasings, the
neighbour capitals, and the broad control set with France + neighbours removed.

In [ ]:
P = marv.Probe
NEIGHBOURS = ['Italy', 'Spain', 'Germany', 'Portugal', 'Belgium']

narrow = [
    P('The capital of France is', 'Paris', ('target',)),
    P('Paris is the capital of', 'France', ('target',)),
    P('The capital of Italy is', 'Rome', ('neighbour',)),
    P('The capital of Spain is', 'Madrid', ('neighbour',)),
    P('The capital of Japan is', 'Tokyo', ('control',)),
    P('Water is made of hydrogen and', 'oxygen', ('control',)),
    P('The opposite of hot is', 'cold', ('control',)),
    P('Two plus two equals', 'four', ('control',)),
    P('The cat sat on the', 'mat', ('control',)),
    P('The sky is', 'blue', ('control',)),
    P('The past tense of go is', 'went', ('control',)),
    P('The chemical symbol for gold is', 'Au', ('control',)),
]

wide = marv.capital_edit_battery('France', 'Paris', neighbours=NEIGHBOURS)
print('narrow:', len(narrow), ' wide:', len(wide))

## Baseline filter

An edit eval is meaningless on facts the model gets wrong unedited -- drop
anything not in the model's top 3. On a 135M base model a fair number of the
harder controls won't survive; what's left is still far more than 8.

In [ ]:
def keep_known(battery, rank_max=3):
    r = marv.run_battery(model, tok, battery, device=device)
    known = {row.prompt for row in r.rows if row.target_rank <= rank_max}
    return [p for p in battery if p.prompt in known]

narrow_k = keep_known(narrow)
wide_k = keep_known(wide)
from collections import Counter
print('narrow kept', len(narrow_k), '/', len(narrow))
print('wide   kept', len(wide_k), '/', len(wide),
      ' controls by sub-domain:',
      dict(Counter(t for p in wide_k for t in p.tags if t not in ('control','target','neighbour','capital'))))

## The causal constellation, then one edit

In [ ]:
target_probes = [p for p in wide_k if 'target' in p.tags]
ranked = marv.rank_by_ablation_effect(
    model, tok, [(r.layer, r.feature) for r in pool[:30]], target_probes, device=device)
for (L, f), drop in ranked[:12]:
    tks, _ = marv.describe_feature(vindex, L, f, k=3)
    print(f'  L{L:>2} f{f:<5} drop={drop:+.3f}  -> {[w.strip() for w in tok.batch_decode([[int(t)] for t in tks])]}')
feats = [c for c, _ in ranked]

In [ ]:
def collateral_line(rep, tag):
    m = rep.metrics().get(tag, {})
    n = int(m.get('n', 0)); p = m.get('moved', 0.0); dp = m.get('mean_dprob', 0.0)
    se = math.sqrt(p * (1 - p) / n) if n else float('nan')
    print(f'  {tag:<10} n={n:>3}  collateral rate {p:.3f} +/- {se:.3f} (1 s.e.)   mean dprob {dp:+.3f}')

for label, batt in [('NARROW', narrow_k), ('WIDE', wide_k)]:
    rep = marv.study_edit(model, tok, marv.suppress(model, feats[:5]), batt, device=device)
    print(f'--- {label} battery, suppress top 5 ---')
    collateral_line(rep, 'target')
    collateral_line(rep, 'neighbour')
    collateral_line(rep, 'control')
    print()

The `control` error bar collapses from ~0.15 to ~0.04 going narrow -> wide.
With the wide battery you can also read collateral **by sub-domain** -- an edit
that quietly damages lexical knowledge looks very different from one that only
nudges geography.

In [ ]:
rep = marv.study_edit(model, tok, marv.suppress(model, feats[:5]), wide_k, device=device)
rep.show()
print()
for tag, m in sorted(rep.metrics().items()):
    if tag == '_all':
        continue
    print(f'  {tag:<14} n={int(m["n"]):>3}  moved={m["moved"]:.3f}  mean dprob={m["mean_dprob"]:+.3f}')

## Which layers carry the fact?

`suppression_by_layer` suppresses one layer's slice of the constellation at a
time. Layers with a `target` drop are load-bearing; layers with features in the
constellation but ~zero effect were geometric KNN hits.

In [ ]:
from marv.evaluate import suppression_by_layer
edit_feats = feats[:8]

print(f'{"L":>3} {"n":>2} {"features":>16}  {"target dp":>10} {"neigh dp":>10} {"ctrl dp":>10}')
iso = suppression_by_layer(model, tok, edit_feats, wide_k, device=device)
for L, fs, d in iso:
    m = d.metrics(); g = lambda t: m.get(t, {}).get('mean_dprob', 0.0)
    print(f'{L:>3} {len(fs):>2} {str(list(fs)):>16}  {g("target"):>+10.3f} {g("neighbour"):>+10.3f} {g("control"):>+10.3f}')

print('\ncumulative, shallow -> deep:')
for L, fs, d in suppression_by_layer(model, tok, edit_feats, wide_k, device=device, cumulative=True):
    m = d.metrics(); g = lambda t: m.get(t, {}).get('mean_dprob', 0.0)
    print(f'  <=L{L:<2}  target {g("target"):>+.3f}   neigh {g("neighbour"):>+.3f}   ctrl {g("control"):>+.3f}')

## The Pareto frontier

In [ ]:
import matplotlib.pyplot as plt
from marv.evaluate import frontier_table

sizes = [0, 1, 2, 3, 4, 6, 8, 10, 14]
sweep = marv.suppression_frontier(model, tok, feats, wide_k, sizes=sizes, device=device)
rows = list(frontier_table(sweep, target='target', collateral='neighbour'))
print(f'{"n":>3} {"target drop":>12} {"neigh drop":>12} {"neigh moved":>12}')
for n, td, cd, cm in rows:
    print(f'{n:>3} {td:>+12.3f} {cd:>+12.3f} {cm:>12.2f}')

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([r[2] for r in rows], [r[1] for r in rows], 'o-')
for n, td, cd, cm in rows:
    ax.annotate(str(n), (cd, td), fontsize=8, xytext=(3, 3), textcoords='offset points')
ax.set_xlabel('neighbour prob drop (collateral)'); ax.set_ylabel('target prob drop (efficacy)')
ax.set_title('AMD-135M: France->Paris suppression frontier'); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Weight-space diff: base vs code finetune

`amd/AMD-Llama-135m-code` is the same 12-layer model finetuned on StarCoder
Python (20B tokens). `marv.diff` shows which FFN features that moved.

In [ ]:
del model; gc.collect()
if device == 'cuda':
    torch.cuda.empty_cache()

code_m = LlamaForCausalLM.from_pretrained(CODE, torch_dtype=torch.float32)
vindex_code = marv.extract(code_m, model_name=CODE)
marv.build_down_meta(vindex_code, device=device)
vindex_code.save('/content/amd-135m-code.vindex.npz')
del code_m; gc.collect()

deltas = marv.diff(vindex, vindex_code)
print('most-moved features, code finetuning:')
for d in marv.most_changed(deltas, k=12):
    b, _ = marv.describe_feature(vindex,      d.layer, d.feature_idx, k=3)
    a, _ = marv.describe_feature(vindex_code, d.layer, d.feature_idx, k=3)
    bw = [w.strip() for w in tok.batch_decode([[int(t)] for t in b])]
    aw = [w.strip() for w in tok.batch_decode([[int(t)] for t in a])]
    print(f'  L{d.layer:>2} f{d.feature_idx:<5} gate_cos={d.gate_cos_sim:+.3f} down_cos={d.down_cos_sim:+.3f}  {bw} -> {aw}')

scores = marv.per_layer_score(deltas, metric='mean_topk')
print('\nlayers that absorbed the most change:')
for L in sorted(scores, key=lambda l: -scores[l])[:6]:
    print(f'  L{L:>2}: {scores[L]:.4f}')

## What else you can do here

- **Swap the fact.** `marv.capital_edit_battery('Japan', 'Tokyo', neighbours=['China','Thailand','Vietnam'])`
  -- or edit a non-capital fact by hand-writing the target/neighbour probes.
- **Rare vs common.** Run the frontier for `France` and for a rarely-mentioned
  country; the rare one usually has a sharper knee (fewer shared features).
- **suppress vs ablate vs steer** on the same constellation (see the Qwen2.5 notebook).
- **`marv.diff(vindex, quantized_vindex)`** -- which features 4-bit breaks (needs a
  bitsandbytes NF4 load; AMD-135M is small enough to hold both copies anywhere).
- Reload any vindex with `marv.VindexLite.load('/content/amd-135m-*.vindex.npz')`.